# Notebook D — Extension experiments for NeurIPS ED Track

**Prereqs:** Notebooks A, B, C complete (E00–E11 in `all_results.csv`, `E00_baseline.csv` saved).

**Runtime estimates (RTX A5000 Pro 24 GB, matches your Jarvis setup):**

| Cell | What runs | GPU-h | Priority |
|---|---|---|---|
| 3 | H1 proximity rerun (CPU-only) | 0 | **MUST for WCTB** |
| 4 | Multi-seed E01–E04 at seed=7 | ~6 | **MUST for ED** |
| 5 | Multi-seed E01–E04 at seed=13 (optional) | ~6 | Nice |
| 6 | Llama-3.1-8B rank ablation E01b–E04b | ~7 | **MUST for ED** |
| 7 | Self-gen dose-response (25 / 50 / 200) | ~5 | **MUST for ED** |
| 8 | LLM-judge quality audit on E11 self-gen | ~0.5 | Nice |
| 9 | MedQA downstream on E02, E03, E10, E11 | ~2 | **MUST for ED** |

**Order to run:** 3 → 4 → 9 → 6 → 7 → 8 → 5. Each cell is idempotent — resume-guards skip already-completed runs.


https://www.kaggle.com/code/akankshanarula/minilm-validations

## Cell 0 — Install dependencies (same pins as Notebook A)

In [11]:
# --- CELL 0: Install ---
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/huggingface/transformers.git"
])

pip("peft>=0.12.0", "trl>=0.10.0",
    "bitsandbytes>=0.43.0", "accelerate>=0.33.0",
    "datasets>=2.20.0", "sentence-transformers>=3.0.0",
    "scipy", "scikit-learn", "matplotlib", "seaborn",
    "openai>=1.30.0")   # for LLM-judge cell

print("Dependencies installed.")

Dependencies installed.


## Cell 1 — Mount Drive and shared config

In [13]:
import os, torch, random, re, json
import numpy as np
import pandas as pd
from datetime import datetime

DRIVE_BASE = "/kaggle/working/"
RESULTS_CSV  = f"/kaggle/input/datasets/akankshanarula/lora-forgetting-final/all_results.csv"
BASELINE_CSV = f"/kaggle/input/datasets/akankshanarula/lora-forgetting-final/E00_baseline.csv"
ADAPTER_DIR  = f"/kaggle/working/adapters"
os.makedirs(f"/kaggle/working/results", exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Qwen from A/B/C
MODEL_QWEN_9B  = "Qwen/Qwen3.5-9B"
# NEW for Notebook D
MODEL_LLAMA_8B = "meta-llama/Llama-3.1-8B-Instruct"   # requires HF token with license access

SEED_PRIMARY = 42
SEED_7       = 7
SEED_13      = 13

MMLU_SUBJECTS = [
    "abstract_algebra","anatomy","astronomy","business_ethics",
    "clinical_knowledge","college_biology","college_chemistry",
    "college_computer_science","college_mathematics","college_medicine",
    "college_physics","computer_security","conceptual_physics",
    "econometrics","electrical_engineering","elementary_mathematics",
    "formal_logic","global_facts","high_school_biology",
    "high_school_chemistry","high_school_computer_science",
    "high_school_european_history","high_school_geography",
    "high_school_government_and_politics","high_school_macroeconomics",
    "high_school_mathematics","high_school_microeconomics",
    "high_school_physics","high_school_psychology","high_school_statistics",
    "high_school_us_history","high_school_world_history","human_aging",
    "human_sexuality","international_law","jurisprudence","logical_fallacies",
    "machine_learning","management","marketing","medical_genetics",
    "miscellaneous","moral_disputes","moral_scenarios","nutrition",
    "philosophy","prehistory","professional_accounting","professional_law",
    "professional_medicine","professional_psychology","public_relations",
    "security_studies","sociology","us_foreign_policy","virology",
    "world_religions",
]
assert len(MMLU_SUBJECTS) == 57

MEDICAL_MMLU = {
    "clinical_knowledge","medical_genetics","college_medicine","anatomy",
    "professional_medicine","virology","nutrition","human_aging",
    "human_sexuality",
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print("Config loaded.")

Config loaded.


## Cell 2 — Shared utilities (copied verbatim from Notebooks A/C for self-containment)

In [14]:
# --- CELL 2: Utilities ---
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset

def log_result(exp_id, result_dict):
    row = {"exp_id": exp_id, "timestamp": datetime.now().isoformat(), **result_dict}
    df = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        df.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df.to_csv(RESULTS_CSV, index=False)

def load_model_4bit(model_id):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True,
    )
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    model.config.use_cache = False
    return model, tok

def eval_mmlu_subject(model, tokenizer, subject, max_samples=None):
    try:
        ds = load_dataset("cais/mmlu", subject, split="test", trust_remote_code=True)
    except Exception as e:
        print(f"  WARN mmlu/{subject}: {e}"); return float("nan")
    if max_samples: ds = ds.select(range(min(max_samples, len(ds))))
    correct, total = 0, 0
    for ex in ds:
        choices_str = "\n".join([f"{l}) {c}" for l,c in zip("ABCD", ex["choices"])])
        prompt = (f"The following is a multiple choice question. "
                  f"Answer with only the letter A, B, C, or D.\n\n"
                  f"Question: {ex['question']}\n{choices_str}\n\nAnswer:")
        msgs = [{"role":"user","content":prompt}]
        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True,
            )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=5, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        m = re.search(r"[ABCD]", gen.upper())
        pred = m.group(0) if m else None
        label = "ABCD"[ex["answer"]]
        correct += int(pred == label); total += 1
    return correct / max(total, 1)

def eval_all_mmlu(model, tokenizer, verbose=True):
    accs = {}
    for i, s in enumerate(MMLU_SUBJECTS):
        accs[s] = eval_mmlu_subject(model, tokenizer, s)
        if verbose and (i+1) % 5 == 0:
            print(f"  [{i+1}/57] {s}: {accs[s]:.3f}")
    return accs

def get_stratified_medqa(n_total=4000, seed=42):
    ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="train", trust_remote_code=True)
    df = ds.to_pandas()
    def get_options(row):
        if "option_0" in df.columns:
            return [row[f"option_{i}"] for i in range(4)]
        elif "options" in df.columns:
            opts = row["options"]
            if isinstance(opts, dict):
                return [opts[k] for k in sorted(opts.keys())[:4]]
            elif isinstance(opts, list):
                return opts[:4]
        return ["A","B","C","D"]
    if "meta_info" in df.columns and df["meta_info"].nunique() > 1:
        n_cats = df["meta_info"].nunique()
        n_per = max(500, n_total // n_cats)
        sampled = df.groupby("meta_info", group_keys=False).apply(
            lambda x: x.sample(min(len(x), n_per), random_state=seed))
    else:
        sampled = df.sample(n=min(n_total, len(df)), random_state=seed)
    rows = []
    for _, r in sampled.iterrows():
        opts = get_options(r)
        q = r.get("question","")
        ans_idx = r.get("answer_idx", r.get("answer", 0))
        if isinstance(ans_idx, str) and ans_idx in "ABCD":
            ans_idx = "ABCD".index(ans_idx)
        choices_str = "\n".join([f"{l}) {c}" for l,c in zip("ABCD", opts)])
        text = (f"The following is a multiple choice question. "
                f"Answer with only the letter A, B, C, or D.\n\n"
                f"Question: {q}\n{choices_str}\n\nAnswer: {'ABCD'[ans_idx]}")
        rows.append({"text": text})
    return Dataset.from_pandas(pd.DataFrame(rows))

def run_finetune(base_model_id, train_dataset, exp_id, lora_rank=16,
                 n_steps=500, seed=42, extra_target_modules=None):
    set_seed(seed)
    model, tok = load_model_4bit(base_model_id)
    target_modules = ["q_proj","k_proj","v_proj","o_proj"]
    if extra_target_modules:
        target_modules += extra_target_modules
    else:
        target_modules += ["gate_proj","up_proj"]
    lora_cfg = LoraConfig(
        r=lora_rank, lora_alpha=lora_rank, lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM, target_modules=target_modules,
    )
    model = get_peft_model(model, lora_cfg)
    sft_cfg = SFTConfig(
        output_dir=f"{ADAPTER_DIR}/{exp_id}",
        num_train_epochs=1,
        max_steps=n_steps,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        logging_steps=50,
        save_strategy="no",
        seed=seed,
        fp16=True,
        dataset_text_field="text",
        max_seq_length=512,
        report_to="none",
    )
    trainer = SFTTrainer(
        model=model, args=sft_cfg,
        train_dataset=train_dataset, tokenizer=tok,
    )
    trainer.train()
    model.save_pretrained(f"{ADAPTER_DIR}/{exp_id}")
    return model, tok

print("Utilities loaded.")

Utilities loaded.


## Cell 3 — H1 MiniLM proximity rerun across ALL ranks (CPU-only)

**Priority: MUST for WCTB submission.**

The workshop draft had a pending proximity correlation — only E10 ran the stats.
This cell computes per-subject MiniLM similarity against the MedQA domain description,
then correlates with forgetting at each rank condition. Output table goes into §4.3
of the paper.

Runtime: ~5 minutes on any machine, no GPU needed. Works offline once MiniLM downloads.


In [15]:
# --- CELL 3: H1 proximity rerun, all ranks ---
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr, ttest_ind

DOMAIN_DESCRIPTIONS = {
    "medqa": ("medicine clinical knowledge anatomy pharmacology pathology "
              "diagnosis treatment disease symptoms physiology"),
}

print("Loading MiniLM (CPU)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
domain_vec = embedder.encode(DOMAIN_DESCRIPTIONS["medqa"], normalize_embeddings=True)
subject_texts = [s.replace("_"," ") for s in MMLU_SUBJECTS]
subject_vecs = embedder.encode(subject_texts, normalize_embeddings=True,
                               batch_size=32, show_progress_bar=False)
proximity = {s: float(np.dot(domain_vec, v)) for s, v in zip(MMLU_SUBJECTS, subject_vecs)}
prox_df = pd.DataFrame(proximity.items(), columns=["subject","proximity"])
prox_df.to_csv(f"/kaggle/working/results/h1_proximity_scores.csv", index=False)
print(f"Saved proximity scores: {prox_df.shape}")

# Load master results and compute per-rank correlations
all_df = pd.read_csv(RESULTS_CSV)

rows = []
TARGET_EXPS = ["E01","E02","E03","E04","E06","E07","E08","E09","E10","E11"]
for exp_id in TARGET_EXPS:
    sub = all_df[all_df["exp_id"] == exp_id].copy()
    if len(sub) < 50: 
        print(f"  skip {exp_id}: only {len(sub)} rows"); continue
    sub["proximity"] = sub["subject"].map(proximity)
    sub = sub.dropna(subset=["proximity","forgetting"])
    r_p, p_p = pearsonr(sub["proximity"], sub["forgetting"])
    r_s, p_s = spearmanr(sub["proximity"], sub["forgetting"])
    # H3 proximity gap: proximal (top-third) vs distal (bottom-third) by proximity
    sub_sorted = sub.sort_values("proximity")
    third = len(sub_sorted) // 3
    distal_mean = sub_sorted.iloc[:third]["forgetting"].mean()
    proximal_mean = sub_sorted.iloc[-third:]["forgetting"].mean()
    t, p_t = ttest_ind(sub_sorted.iloc[-third:]["forgetting"],
                       sub_sorted.iloc[:third]["forgetting"])
    rows.append({
        "exp_id": exp_id,
        "n_subjects": len(sub),
        "pearson_r": r_p, "pearson_p": p_p,
        "spearman_r": r_s, "spearman_p": p_s,
        "proximal_mean_f": proximal_mean,
        "distal_mean_f": distal_mean,
        "proximity_gap": proximal_mean - distal_mean,
        "t_stat": t, "t_p": p_t,
    })

prox_results = pd.DataFrame(rows)
prox_results.to_csv(f"/kaggle/working//results/h1_proximity_per_exp.csv", index=False)
print("\nH1 proximity analysis per experiment:")
print(prox_results.round(3).to_string(index=False))

Loading MiniLM (CPU)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Saved proximity scores: (57, 2)

H1 proximity analysis per experiment:
exp_id  n_subjects  pearson_r  pearson_p  spearman_r  spearman_p  proximal_mean_f  distal_mean_f  proximity_gap  t_stat   t_p
   E01          57      0.065      0.634       0.039       0.773            0.162          0.144          0.019   0.892 0.378
   E02          57     -0.036      0.788      -0.173       0.198            0.351          0.369         -0.018  -0.522 0.605
   E03          57      0.035      0.793      -0.053       0.694            0.555          0.557         -0.002  -0.039 0.969
   E04          57      0.045      0.738      -0.047       0.730            0.562          0.559          0.003   0.056 0.955
   E06          57     -0.002      0.986      -0.151       0.263            0.444          0.453         -0.009  -0.224 0.824
   E07          57     -0.010      0.940      -0.160       0.236            0.455          0.467         -0.011  -0.277 0.784
   E08          57      0.079      0.557      -

## Cell 4 — Multi-seed repeat of E01–E04 at seed=7

**Priority: MUST for NeurIPS ED.** Produces error bars for the rank ablation.
Expected runtime: ~6 GPU-hours on A5000.

The resume guard skips any E01_s7/E02_s7/... already present in `all_results.csv`.


In [ ]:
# --- CELL 4: seed=7 repeat of rank ablation ---
baseline_df = pd.read_csv(BASELINE_CSV)
baseline_accs = dict(zip(baseline_df["subject"], baseline_df["accuracy"]))

existing = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
done = set(existing["exp_id"].unique()) if len(existing) else set()

medqa_train = get_stratified_medqa(n_total=4000, seed=SEED_7)
print(f"MedQA loaded: {len(medqa_train)} examples")

RANK_REPEATS_S7 = [
    ("E01_s7",   4, 500),
    ("E02_s7",  16, 500),
    ("E03_s7",  64, 500),
    ("E04_s7", 128, 500),
]

for exp_id, rank, steps in RANK_REPEATS_S7:
    if exp_id in done:
        print(f"[skip] {exp_id} already in results"); continue
    print(f"\n{'='*60}\n  {exp_id}: rank={rank}, seed=7\n{'='*60}")
    ft_model, ft_tok = run_finetune(
        MODEL_QWEN_9B, medqa_train, exp_id,
        lora_rank=rank, n_steps=steps, seed=SEED_7,
    )
    ft_model.eval()
    post_accs = eval_all_mmlu(ft_model, ft_tok)
    for subj in MMLU_SUBJECTS:
        b = baseline_accs.get(subj, float("nan"))
        p = post_accs.get(subj, float("nan"))
        log_result(exp_id, {
            "model":"Qwen3.5-9B","domain":"medqa","lora_rank":rank,
            "n_steps":steps,"replay_size":0,"seed":SEED_7,
            "subject":subj,"is_medical":subj in MEDICAL_MMLU,
            "accuracy":p,"delta_acc":p-b,"forgetting":b-p,
        })
    del ft_model, ft_tok; torch.cuda.empty_cache()
    import gc; gc.collect()
    print(f"[done] {exp_id}")

## Cell 5 — (Optional) seed=13 repeat. Skip unless GPU budget remains.

In [ ]:
# --- CELL 5: seed=13 repeat (only if time permits) ---
RANK_REPEATS_S13 = [
    ("E01_s13",   4, 500),
    ("E02_s13",  16, 500),
    ("E03_s13",  64, 500),
    ("E04_s13", 128, 500),
]

existing = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
done = set(existing["exp_id"].unique()) if len(existing) else set()
medqa_train_s13 = get_stratified_medqa(n_total=4000, seed=SEED_13)

for exp_id, rank, steps in RANK_REPEATS_S13:
    if exp_id in done:
        print(f"[skip] {exp_id}"); continue
    print(f"\n{'='*60}\n  {exp_id}\n{'='*60}")
    ft_model, ft_tok = run_finetune(
        MODEL_QWEN_9B, medqa_train_s13, exp_id,
        lora_rank=rank, n_steps=steps, seed=SEED_13,
    )
    ft_model.eval()
    post = eval_all_mmlu(ft_model, ft_tok)
    for s in MMLU_SUBJECTS:
        b = baseline_accs.get(s, float("nan")); p = post.get(s, float("nan"))
        log_result(exp_id, {
            "model":"Qwen3.5-9B","domain":"medqa","lora_rank":rank,
            "n_steps":steps,"replay_size":0,"seed":SEED_13,
            "subject":s,"is_medical":s in MEDICAL_MMLU,
            "accuracy":p,"delta_acc":p-b,"forgetting":b-p,
        })
    del ft_model, ft_tok; torch.cuda.empty_cache()
    import gc; gc.collect()

## Cell 6 — Llama-3.1-8B cross-family rank ablation (E01b–E04b)

**Priority: MUST for NeurIPS ED.** Cross-family replication is the single biggest
credibility booster for ED reviewers. ~7 GPU-hours.

**Prereq:** You need an HF token with `meta-llama/Llama-3.1-8B-Instruct` access.
Run `from huggingface_hub import login; login()` and paste your token before this cell.

This first evaluates Llama baseline on all 57 MMLU subjects (so we can compute
forgetting relative to Llama's own baseline, not Qwen's). Then fine-tunes at 4 ranks.


In [ ]:
# --- CELL 6a: Llama baseline E00b ---
LLAMA_BASELINE_CSV = f"{DRIVE_BASE}/results/E00b_llama_baseline.csv"

if not os.path.exists(LLAMA_BASELINE_CSV):
    print("Running Llama-3.1-8B baseline on all 57 subjects...")
    lmodel, ltok = load_model_4bit(MODEL_LLAMA_8B)
    lmodel.eval()
    llama_base = {}
    for i, subj in enumerate(MMLU_SUBJECTS):
        llama_base[subj] = eval_mmlu_subject(lmodel, ltok, subj)
        if (i+1) % 5 == 0:
            pd.DataFrame([{"subject":s,"accuracy":a,"is_medical":s in MEDICAL_MMLU}
                          for s,a in llama_base.items()]).to_csv(LLAMA_BASELINE_CSV, index=False)
            print(f"  [{i+1}/57] baseline saved")
    pd.DataFrame([{"subject":s,"accuracy":a,"is_medical":s in MEDICAL_MMLU}
                  for s,a in llama_base.items()]).to_csv(LLAMA_BASELINE_CSV, index=False)
    del lmodel, ltok; torch.cuda.empty_cache()
    import gc; gc.collect()
    print("Llama baseline done.")
else:
    print(f"Llama baseline already exists: {LLAMA_BASELINE_CSV}")

llama_base_df = pd.read_csv(LLAMA_BASELINE_CSV)
llama_baseline = dict(zip(llama_base_df["subject"], llama_base_df["accuracy"]))

In [ ]:
# --- CELL 6b: Llama rank ablation E01b–E04b ---
LLAMA_RANK_EXPS = [
    ("E01b",   4, 500),
    ("E02b",  16, 500),
    ("E03b",  64, 500),
    ("E04b", 128, 500),
]

existing = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
done = set(existing["exp_id"].unique()) if len(existing) else set()
medqa_train = get_stratified_medqa(n_total=4000, seed=SEED_PRIMARY)

for exp_id, rank, steps in LLAMA_RANK_EXPS:
    if exp_id in done:
        print(f"[skip] {exp_id}"); continue
    print(f"\n{'='*60}\n  {exp_id}: Llama r={rank}\n{'='*60}")
    ft_model, ft_tok = run_finetune(
        MODEL_LLAMA_8B, medqa_train, exp_id,
        lora_rank=rank, n_steps=steps, seed=SEED_PRIMARY,
    )
    ft_model.eval()
    post = eval_all_mmlu(ft_model, ft_tok)
    for s in MMLU_SUBJECTS:
        b = llama_baseline.get(s, float("nan")); p = post.get(s, float("nan"))
        log_result(exp_id, {
            "model":"Llama-3.1-8B","domain":"medqa","lora_rank":rank,
            "n_steps":steps,"replay_size":0,"seed":SEED_PRIMARY,
            "subject":s,"is_medical":s in MEDICAL_MMLU,
            "accuracy":p,"delta_acc":p-b,"forgetting":b-p,
        })
    del ft_model, ft_tok; torch.cuda.empty_cache()
    import gc; gc.collect()
    print(f"[done] {exp_id}")

## Cell 7 — Self-gen replay dose-response: 25, 50, 200 samples

**Priority: MUST for NeurIPS ED.** Turns E11 from a single point into a curve.
E10 (100 real), E11 (100 self-gen) already done. This adds 25/50/200 self-gen.
~5 GPU-hours.

The `build_self_generated_replay` helper is in Notebook C Cell 5 — if you need it
re-executed, copy that cell here or import from the same Drive folder.


In [ ]:
# --- CELL 7a: Build / reuse the 200-sample self-gen buffer ---
# We reuse Notebook C's buffer if present; else build once with n=200.
BUFFER_PATH_200 = f"{DRIVE_BASE}/results/self_gen_buffer_s42_n200.jsonl"
BUFFER_PATH_OLD = f"{DRIVE_BASE}/results/self_gen_buffer_s42.jsonl"  # Notebook C, n=100

def build_self_generated_replay(model, tokenizer, n_examples=200, seed=42):
    """Same spirit as Notebook C Cell 5 but extended to 200 samples."""
    set_seed(seed)
    TOPICS = [
        "cardiology","endocrinology","neurology","oncology","pharmacology",
        "pathology","immunology","psychiatry","pediatrics","surgery",
        "radiology","nephrology","hepatology","dermatology","ophthalmology",
        "obstetrics","anesthesiology","emergency medicine","infectious disease",
        "rheumatology",
    ]
    out = []
    per_topic = max(1, n_examples // len(TOPICS) + 1)
    for t in TOPICS:
        for _ in range(per_topic):
            prompt = (f"Write a board-style USMLE multiple choice question about {t}. "
                      f"Format:\nQuestion: ...\nA) ...\nB) ...\nC) ...\nD) ...\n"
                      f"Answer: X")
            msgs = [{"role":"user","content":prompt}]
            text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                 add_generation_prompt=True)
            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            with torch.no_grad():
                gen = model.generate(**inputs, max_new_tokens=300,
                                     do_sample=True, top_p=0.95, temperature=0.8,
                                     pad_token_id=tokenizer.pad_token_id)
            txt = tokenizer.decode(gen[0][inputs.input_ids.shape[1]:],
                                   skip_special_tokens=True)
            if "Answer:" in txt and any(f"{L})" in txt for L in "ABCD"):
                out.append({"text": txt.strip(), "topic": t})
                if len(out) >= n_examples:
                    return out
    return out

if not os.path.exists(BUFFER_PATH_200):
    print("Building 200-sample self-gen buffer (one-shot)...")
    m, t = load_model_4bit(MODEL_QWEN_9B); m.eval()
    buf = build_self_generated_replay(m, t, n_examples=200, seed=SEED_PRIMARY)
    with open(BUFFER_PATH_200,"w") as f:
        for ex in buf: f.write(json.dumps(ex)+"\n")
    del m, t; torch.cuda.empty_cache()
    import gc; gc.collect()
    print(f"Buffer saved: {len(buf)} examples -> {BUFFER_PATH_200}")
else:
    print(f"Buffer exists: {BUFFER_PATH_200}")

# Load buffer into memory
with open(BUFFER_PATH_200) as f:
    sg_buffer = [json.loads(l) for l in f if l.strip()]
print(f"Loaded {len(sg_buffer)} self-gen examples.")

In [ ]:
# --- CELL 7b: Dose-response runs E11a/E11b/E11c at 25/50/200 ---
from datasets import Dataset as HFDataset

existing = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
done = set(existing["exp_id"].unique()) if len(existing) else set()
baseline_df = pd.read_csv(BASELINE_CSV)
baseline_accs = dict(zip(baseline_df["subject"], baseline_df["accuracy"]))

medqa_train = get_stratified_medqa(n_total=4000, seed=SEED_PRIMARY)

DOSE_EXPS = [
    ("E11a",  25),
    ("E11b",  50),
    ("E11c", 200),
]

for exp_id, n_replay in DOSE_EXPS:
    if exp_id in done:
        print(f"[skip] {exp_id}"); continue
    print(f"\n{'='*60}\n  {exp_id}: self-gen n={n_replay}\n{'='*60}")

    # Mix MedQA + N self-gen. Buffer is pre-shuffled by topic so slice is OK.
    replay_slice = sg_buffer[:n_replay]
    combined = list(medqa_train) + [{"text": x["text"]} for x in replay_slice]
    random.Random(SEED_PRIMARY).shuffle(combined)
    mixed = HFDataset.from_pandas(pd.DataFrame(combined))

    ft_model, ft_tok = run_finetune(
        MODEL_QWEN_9B, mixed, exp_id,
        lora_rank=16, n_steps=500, seed=SEED_PRIMARY,
    )
    ft_model.eval()
    post = eval_all_mmlu(ft_model, ft_tok)
    for s in MMLU_SUBJECTS:
        b = baseline_accs.get(s, float("nan")); p = post.get(s, float("nan"))
        log_result(exp_id, {
            "model":"Qwen3.5-9B","domain":"medqa+selfgen","lora_rank":16,
            "n_steps":500,"replay_size":n_replay,"seed":SEED_PRIMARY,
            "subject":s,"is_medical":s in MEDICAL_MMLU,
            "accuracy":p,"delta_acc":p-b,"forgetting":b-p,
        })
    del ft_model, ft_tok; torch.cuda.empty_cache()
    import gc; gc.collect()
    print(f"[done] {exp_id}")

## Cell 8 — LLM-judge quality audit on E11 self-gen buffer

**Priority: nice-to-have.** Addresses reviewer point "E11 unaudited." 
Runs a strong judge (GPT-4o-mini by default) over 50 random samples from the
self-gen buffer, scoring each on a 1–5 rubric (well-formed question, single
correct answer, medically plausible, answer matches). Reports pass-rate.

Requires `OPENAI_API_KEY`. If you don't have one, skip and note this is future
work in §6.


In [ ]:
# --- CELL 8: LLM-judge quality audit ---
import openai, random
# Set API key before running:
# os.environ["OPENAI_API_KEY"] = "sk-..."

client = openai.OpenAI()
JUDGE_MODEL = "gpt-4o-mini"

RUBRIC = """Rate this USMLE-style question on four criteria (1-5 each):
1. well_formed: Is the question grammatical and complete?
2. single_answer: Is there exactly one correct answer among A-D?
3. medically_plausible: Is the clinical scenario realistic and the answer medically correct?
4. answer_matches: Does the stated answer letter match the correct option?

Respond with ONLY a JSON object:
{"well_formed": N, "single_answer": N, "medically_plausible": N, "answer_matches": N}"""

def judge_one(question_text):
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role":"system","content":"You are a medical educator grading exam questions."},
            {"role":"user","content":RUBRIC + "\n\nQUESTION:\n" + question_text},
        ],
        temperature=0,
        response_format={"type":"json_object"},
    )
    return json.loads(resp.choices[0].message.content)

random.seed(42)
sample = random.sample(sg_buffer, min(50, len(sg_buffer)))
scores = []
for i, ex in enumerate(sample):
    try:
        s = judge_one(ex["text"])
        s["idx"] = i; scores.append(s)
    except Exception as e:
        print(f"  judge error {i}: {e}")

judge_df = pd.DataFrame(scores)
judge_df.to_csv(f"{DRIVE_BASE}/results/e11_quality_audit.csv", index=False)
print("\nLLM-judge audit summary:")
for col in ["well_formed","single_answer","medically_plausible","answer_matches"]:
    print(f"  {col}: mean={judge_df[col].mean():.2f}  pass@4+={sum(judge_df[col]>=4)/len(judge_df):.1%}")
print(f"\nOverall pass rate (all 4 >= 4): "
      f"{(judge_df[['well_formed','single_answer','medically_plausible','answer_matches']].min(axis=1)>=4).mean():.1%}")

## Cell 9 — MedQA downstream accuracy on E02, E03, E10, E11 adapters

**Priority: MUST for NeurIPS ED.** Kills the "MMLU ≠ clinical" reviewer criticism.
For each adapter, evaluates on held-out MedQA-USMLE test split. ~2 GPU-hours.

Reports per-condition target-task accuracy to be inserted into §4.5.


In [ ]:
# --- CELL 9: MedQA downstream ---
def eval_medqa_test(model, tokenizer, max_samples=500):
    ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="test",
                      trust_remote_code=True).select(range(max_samples))
    correct = 0; total = 0
    for ex in ds:
        opts = ex.get("options")
        if isinstance(opts, dict):
            opts = [opts[k] for k in sorted(opts.keys())[:4]]
        elif not isinstance(opts, list):
            opts = [ex.get(f"option_{i}","") for i in range(4)]
        q = ex["question"]
        a = ex.get("answer_idx", ex.get("answer", 0))
        if isinstance(a, str) and a in "ABCD": a = "ABCD".index(a)
        choices_str = "\n".join([f"{L}) {c}" for L,c in zip("ABCD", opts)])
        prompt = (f"The following is a multiple choice question. "
                  f"Answer with only the letter A, B, C, or D.\n\n"
                  f"Question: {q}\n{choices_str}\n\nAnswer:")
        msgs = [{"role":"user","content":prompt}]
        try:
            text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                 add_generation_prompt=True,
                                                 enable_thinking=False)
        except TypeError:
            text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                 add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=5, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        m = re.search(r"[ABCD]", gen.upper())
        pred = m.group(0) if m else None
        correct += int(pred == "ABCD"[a]); total += 1
    return correct / max(total, 1)

# Also compute for BASE model to give a reference point.
DOWNSTREAM_TARGETS = [
    ("E00_base_qwen", None),            # base model, no adapter
    ("E02", f"{ADAPTER_DIR}/E02"),
    ("E03", f"{ADAPTER_DIR}/E03"),
    ("E10", f"{ADAPTER_DIR}/E10"),
    ("E11", f"{ADAPTER_DIR}/E11"),
]

results = []
for exp_id, adapter_path in DOWNSTREAM_TARGETS:
    print(f"\nDownstream eval: {exp_id}")
    m, t = load_model_4bit(MODEL_QWEN_9B)
    if adapter_path and os.path.exists(adapter_path):
        m = PeftModel.from_pretrained(m, adapter_path)
    m.eval()
    acc = eval_medqa_test(m, t, max_samples=500)
    print(f"  MedQA test acc: {acc:.3f}")
    results.append({"exp_id": exp_id, "medqa_test_acc": acc})
    del m, t; torch.cuda.empty_cache()
    import gc; gc.collect()

ds_df = pd.DataFrame(results)
ds_df.to_csv(f"{DRIVE_BASE}/results/medqa_downstream.csv", index=False)
print("\nDownstream results:")
print(ds_df.to_string(index=False))

## Cell 10 — Consolidate all results + regenerate figures

Builds a single tidy DataFrame suitable for the paper and produces the
4 figures in `paper/figures/`.


In [ ]:
# --- CELL 10: Consolidate + figures ---
import matplotlib.pyplot as plt

FIG_DIR = f"{DRIVE_BASE}/paper/figures"
os.makedirs(FIG_DIR, exist_ok=True)

all_df = pd.read_csv(RESULTS_CSV)
summary = (all_df.groupby(["exp_id","is_medical"])
                  .agg(mean_forgetting=("forgetting","mean"),
                       std_forgetting=("forgetting","std"),
                       n=("subject","count"))
                  .reset_index())
summary.to_csv(f"{DRIVE_BASE}/results/summary_per_exp.csv", index=False)
print("Per-experiment summary saved.")
print(summary)

# Figure 1: rank ablation (seed=42)
fig, ax = plt.subplots(figsize=(4.5,3.2))
ranks = [4,16,64,128]
main_means = [all_df[all_df.exp_id==f"E0{i}"].forgetting.mean() for i in [1,2,3,4]]
ax.plot(ranks, main_means, "o-", color="black", label="Qwen3.5-9B seed=42")
# Overlay seed=7 if present
s7_means = [all_df[all_df.exp_id==f"E0{i}_s7"].forgetting.mean() for i in [1,2,3,4]]
if not any(np.isnan(s7_means)):
    ax.plot(ranks, s7_means, "s--", color="gray", label="Qwen3.5-9B seed=7")
# Overlay Llama if present
llama_means = [all_df[all_df.exp_id==f"E0{i}b"].forgetting.mean() for i in [1,2,3,4]]
if not any(np.isnan(llama_means)):
    ax.plot(ranks, llama_means, "^-", color="C1", label="Llama-3.1-8B seed=42")
ax.set_xlabel("LoRA rank"); ax.set_ylabel("Mean forgetting")
ax.set_xscale("log"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig1_rank.pdf", dpi=300); plt.close()
print("Figure 1 saved.")

# Figure 2: proximity gap by rank
gaps = []
for exp, r in [("E01",4),("E02",16),("E03",64),("E04",128)]:
    sub = all_df[all_df.exp_id==exp]
    if len(sub)==0: continue
    m = sub[sub.is_medical].forgetting.mean()
    nm = sub[~sub.is_medical].forgetting.mean()
    gaps.append((r, m - nm))
gdf = pd.DataFrame(gaps, columns=["rank","gap"])
fig, ax = plt.subplots(figsize=(4.5,3.2))
ax.bar(range(len(gdf)), gdf["gap"], color=["C0" if g<=0 else "C3" for g in gdf["gap"]])
ax.set_xticks(range(len(gdf)))
ax.set_xticklabels([f"r={r}" for r in gdf["rank"]])
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("Proximity gap (medical - non-medical)")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig2_proximity.pdf", dpi=300); plt.close()
print("Figure 2 saved.")

# Figure 3: step dynamics
step_exps = [("E06",100),("E07",200),("E02",500),("E08",1000)]
means = []
for exp, s in step_exps:
    sub = all_df[all_df.exp_id==exp]
    means.append((s, sub.forgetting.mean()))
sdf = pd.DataFrame(means, columns=["steps","forgetting"])
fig, ax = plt.subplots(figsize=(4.5,3.2))
ax.plot(sdf["steps"], sdf["forgetting"], "o-", color="black")
ax.set_xlabel("Training steps (rank=16)"); ax.set_ylabel("Mean forgetting")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig3_steps.pdf", dpi=300); plt.close()
print("Figure 3 saved.")

# Figure 4: replay dose-response
replay_exps = [("E02",0,"no replay"),
               ("E09",50,"50 real"),("E10",100,"100 real"),
               ("E11a",25,"25 self-gen"),("E11b",50,"50 self-gen"),
               ("E11",100,"100 self-gen"),("E11c",200,"200 self-gen")]
rows = []
for exp, n, lbl in replay_exps:
    sub = all_df[all_df.exp_id==exp]
    if len(sub)>0:
        rows.append({"label":lbl, "n":n, "forgetting":sub.forgetting.mean(),
                     "kind":"self-gen" if "self-gen" in lbl else ("real" if "real" in lbl else "none")})
rdf = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(5.5,3.2))
for kind, color in [("real","C0"),("self-gen","C3"),("none","gray")]:
    sub = rdf[rdf["kind"]==kind]
    if len(sub): ax.plot(sub["n"], sub["forgetting"], "o-", color=color, label=kind)
ax.set_xlabel("Replay examples"); ax.set_ylabel("Mean forgetting")
ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig4_replay.pdf", dpi=300); plt.close()
print("Figure 4 saved.")

print("\nAll figures written to", FIG_DIR)

## Cell 11 — Cleanup

In [ ]:
# --- CELL 11: Cleanup ---
import gc
torch.cuda.empty_cache(); gc.collect()
print("Notebook D complete.")
print(f"Master results: {RESULTS_CSV}")
print(f"Figures: {DRIVE_BASE}/paper/figures/")